# B2-019-attention-transformers — Practice p14 — Solution

**Type:** proof · **Difficulty:** core · **Concepts:** causal-self-attention

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

Base case: the layer-zero representation at i is its input at i, hence depends only on positions through i. Induction step: assume every layer-\(\ell\) representation at source j depends only on inputs 0,...,j. Causal row i uses only j<=i, so every attended value depends only on inputs through i. Position-wise sublayers do not mix positions, and adding the residual at i introduces no later dependency; the invariant holds at layer \(\ell+1\). Post-softmax zeroing does not restore causality: a forbidden future value is removed from the numerator, but its key score remains in the softmax denominator and therefore changes every surviving allowed weight. With scores (0,0), allowed mask (1,0), and values (2,999), the post-masked output is 1. Changing only the forbidden future score to 10 makes it approximately 0.0000907957. The surviving weights also sum to 1/2 and approximately 0.0000453979 rather than one. Pre-softmax masking excludes the future score from the denominator, so both outputs are exactly 2. A reversed triangle more directly admits j>i, so future values enter the output as well.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 1e-12
RTOL = 1e-12
allowed = np.array([[1.0, 0.0]], dtype=np.float64)
values = np.array([[2.0], [999.0]], dtype=np.float64)
scores_before = np.array([[0.0, 0.0]], dtype=np.float64)
scores_after = np.array([[0.0, 10.0]], dtype=np.float64)

def row_softmax(scores):
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    numerators = np.exp(shifted)
    return numerators / np.sum(numerators, axis=-1, keepdims=True)

postmasked_weights_before = row_softmax(scores_before) * allowed
postmasked_weights_after = row_softmax(scores_after) * allowed
postmasked_output_before = float((postmasked_weights_before @ values)[0, 0])
postmasked_output_after = float((postmasked_weights_after @ values)[0, 0])
additive_mask = np.where(allowed.astype(bool), 0.0, -np.inf)
premasked_weights_before = row_softmax(scores_before + additive_mask)
premasked_weights_after = row_softmax(scores_after + additive_mask)
premasked_output_before = float((premasked_weights_before @ values)[0, 0])
premasked_output_after = float((premasked_weights_after @ values)[0, 0])
reversed_output = float((np.array([[0.0, 1.0]]) @ values)[0, 0])

### Answer check

In [ ]:
np.testing.assert_allclose(postmasked_weights_before, [[0.5, 0.0]], atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(postmasked_weights_after, [[4.5397868702434395e-5, 0.0]], atol=ATOL, rtol=RTOL)
assert np.isclose(postmasked_output_before, 1.0, atol=ATOL, rtol=RTOL)
assert np.isclose(postmasked_output_after, 9.079573740486879e-5, atol=ATOL, rtol=RTOL)
assert not np.isclose(postmasked_output_before, postmasked_output_after, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(premasked_weights_before, [[1.0, 0.0]], atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(premasked_weights_after, [[1.0, 0.0]], atol=ATOL, rtol=RTOL)
assert np.isclose(premasked_output_before, 2.0, atol=ATOL, rtol=RTOL)
assert np.isclose(premasked_output_after, 2.0, atol=ATOL, rtol=RTOL)
assert np.isclose(reversed_output, 999.0, atol=ATOL, rtol=RTOL)